In [ ]:
dbutils.widgets.dropdown(
    "run_fixture_tests",
    "false",
    ["false", "true"]
)

run_fixture_tests = (
    dbutils.widgets.get("run_fixture_tests") == "true"
)

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

order_items = spark.table(
    f"{catalog}.silver.order_items_current"
)

payments = spark.table(
    f"{catalog}.silver.payments_current"
)

captured_payments = (
    payments
    .filter(
        (F.col("status") == "captured")
    )
)

In [ ]:
# One row per order and currency.

order_totals = (
    order_items
    .groupBy('order_id', "currency")
    .agg(
        F.sum("line_amount").alias("order_total")
    )
)

payment_totals = (
    captured_payments
    .groupBy("order_id" , "currency")
    .agg(
        F.sum("amount").alias("paid_total"),
        F.count('*').alias("payment_count")
    )
)

display(order_totals)
display(payment_totals)

In [ ]:
reconciliation =(
    order_totals
    .join(
        payment_totals,
        on = ["order_id" , "currency"],
        how = "full_outer"
    )
    .withColumn(
        "payment_count",
        F.coalesce(F.col("payment_count") , F.lit(0))
    )
    .withColumn(
        "paid_total",
        F.coalesce(F.col("paid_total") , F.lit(0).cast("decimal(18,2)"))
    )
    .withColumn(
        "balance",
        F.col("order_total") - F.col("paid_total")
    )
    .withColumn(
        "reconciliation_status",
        F.when(
            F.col('order_total').isNull(),
            F.lit("unmatched_payment")
        )
        .when(
            F.col("payment_count") == 0,
            F.lit("unpaid")
        )
        .when(
            F.col("paid_total") < F.col("order_total"),
            F.lit("partially_paid")
        )
        .when(
            F.col("paid_total") == F.col("order_total"),
            F.lit("fully_paid")
        )
        .otherwise(
            F.lit("overpaid")
        )
    )
)

display(
    reconciliation
    .groupBy("reconciliation_status")
    .count()
)

display(
    reconciliation.orderBy("order_id")
)

In [ ]:
# Each order/currency combination must appear once.
duplicate_keys = (
    reconciliation
    .groupBy("order_id", "currency")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_keys.limit(1).count() == 0, (
    "Duplicate order/currency rows exist in reconciliation."
)


# Compare payment totals before and after the join, per currency.
expected_totals = (
    captured_payments
    .groupBy("currency")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.count("*").alias("payment_count"),
    )
)

actual_totals = (
    reconciliation
    .filter(F.col("payment_count") > 0)
    .groupBy("currency")
    .agg(
        F.sum("paid_total").alias("total_amount"),
        F.sum("payment_count").alias("payment_count"),
    )
)

assert expected_totals.exceptAll(actual_totals).count() == 0, (
    "Payment totals changed during reconciliation."
)

assert actual_totals.exceptAll(expected_totals).count() == 0, (
    "Reconciliation contains unexpected payment totals."
)

print("Reconciliation integrity checks passed.")

In [ ]:
if run_fixture_tests:
    from decimal import Decimal

    expected_orders = {
        "O10001": Decimal("50.00"),
        "O-B001-NOT-YET-ARRIVED": Decimal("15.00"),
    }

    for order_id, expected_amount in expected_orders.items():
        result = (
            reconciliation
            .filter(
                (F.col("order_id") == order_id)
                & (F.col("currency") == "GBP")
            )
            .first()
        )

        assert result is not None, (
            f"Missing order: {order_id}"
        )

        assert (
            result["order_total"] == expected_amount
            and result["paid_total"] == expected_amount
            and result["balance"] == Decimal("0.00")
            and result["payment_count"] == 1
            and result["reconciliation_status"] == "fully_paid"
        ), f"Unexpected reconciliation for {order_id}"

    print("Original and late-arriving order fixture checks passed.")

else:
    print("Skipped fixed-value reconciliation fixture tests.")

In [ ]:
reconciliation_table = (
    f"{catalog}.gold.order_payment_reconciliation"
)

(
    reconciliation
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(reconciliation_table)
)

print(f"Saved: {reconciliation_table}")

display(
    spark.table(reconciliation_table)
    .groupBy("reconciliation_status")
    .count()
)